# ISLES'24 Fold 4 handoff V2

CPU only. This notebook creates small resumable 5-subject shards from the existing nnU-Net preprocessed files and Fold 4 checkpoint. It does not upload anything yet.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RESULTS=ROOT/'results'; SHARDS=ROOT/'kaggle_handoff_shards_v2'
SCRATCH=Path('/content/isles2024_kaggle_scratch_v2')
LOG=RESULTS/'kaggle_handoff_v2.log'
ZIP=Path('/content/HybridStrokeSeg-main.zip')
REPO=Path('/content/HybridStrokeSeg-Reproduction-main')
RESULTS.mkdir(parents=True,exist_ok=True); SHARDS.mkdir(parents=True,exist_ok=True); SCRATCH.mkdir(parents=True,exist_ok=True)
def run_logged(cmd,cwd=None):
    cmd=[str(x) for x in cmd]; print('$',' '.join(cmd),flush=True)
    with LOG.open('a',encoding='utf-8') as h:
        p=subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert p.stdout is not None
        for line in p.stdout:
            print(line,end=''); h.write(line); h.flush()
        rc=p.wait()
        if rc: raise subprocess.CalledProcessError(rc,cmd)
LOG.write_text('ISLES24 Kaggle handoff V2 started\n',encoding='utf-8')


In [ ]:
if REPO.exists(): shutil.rmtree(REPO)
if ZIP.exists(): ZIP.unlink()
run_logged(['curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors','https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip','-o',ZIP])
run_logged(['unzip','-q',ZIP,'-d','/content'])
os.chdir(REPO)
run_logged([sys.executable,'-m','pip','install','-q','-e','.'])


In [ ]:
run_logged([sys.executable,'-u','scripts/make_isles2024_kaggle_shards_v2.py','--preprocessed-root',ROOT/'nnUNet_preprocessed','--results-root',ROOT/'nnUNet_results','--output-dir',SHARDS,'--scratch-dir',SCRATCH,'--fold','4','--subjects-per-shard','5'],cwd=REPO)
manifest=SHARDS/'handoff_manifest.json'
payload=json.loads(manifest.read_text())
print('Subjects:',payload['subjects'])
print('Data shards:',len(payload['preprocessed_shards']))
print('Ready:',manifest)


If the runtime disconnects, run the notebook again. Completed shards are skipped. The persistent log is `MyDrive/HybridStrokeSeg/ISLES2024/results/kaggle_handoff_v2.log`.
